# 步驟一：建立自己的 LLM Wiki（AI Agents 主題）

作業：生成式AI 第十週作楪 —— 用 LLM Wiki 做教學 HTML 簡報

本會話執行 **先討論、後生成**的 ingest 流程：從 5 篇 arXiv 論文抓取標題與摘要 → 整理成 `wiki/` 內的知識庫檔案 → 產生 `index.md` 目錄。

In [ ]:
# %%cell【取得資料】ingest 5 篇論文（arXiv）
import urllib.request, re, time, os, json

SOURCES = {
    'react':        'https://arxiv.org/abs/2210.03629',   # ReAct: Reasoning + Acting
    'reflexion':    'https://arxiv.org/abs/2303.11366',   # Reflexion: verbal self-correction loop
    'toolformer':   'https://arxiv.org/abs/2302.04761',   # Toolformer: tool use
    'genagents':    'https://arxiv.org/abs/2304.03442',   # Generative Agents: memory+reflection
    'crag':         'https://arxiv.org/abs/2401.15884',   # CRAG: corrective RAG (check node)
}

os.makedirs('wiki', exist_ok=True)
records = []
for name, url in SOURCES.items():
    html = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent':'Mozilla/5.0'}), timeout=30).read().decode('utf-8', 'ignore')
    title = re.search(r'<meta name="citation_title" content="(.*?)"', html).group(1)
    authors = re.findall(r'<meta name="citation_author" content="(.*?)"', html)[:3]
    date  = re.search(r'<meta name="citation_date" content="(.*?)"', html)
    absm  = re.search(r'<meta name="citation_abstract" content="(.*?)"', html, re.S)
    abstract = re.sub(r'\s+', ' ', absm.group(1)).strip() if absm else ''
    with open(f'wiki/{name}.md', 'w', encoding='utf-8') as f:
        f.write(f'# {title}\n\n- 來源：{url}\n- 作者：{", ".join(authors)}\n- 日期：{date.group(1) if date else ""}\n\n## Abstract\n{abstract}\n\n## Wiki 整理（loop/graph 角色）\n見筆記本討論結論。\n')
    records.append({'name': name, 'title': title, 'url': url})
    print(f'[ingest ok] {name}: {title[:60]}')
    time.sleep(1)

with open('wiki/index.md', 'w', encoding='utf-8') as f:
    f.write('# AI Agents LLM Wiki \u2014 \u76ee\u9304\n\n')
    for r in records:
        f.write(f"- [{r['title']}]({r['url']}) \uff08wiki/{r['name']}.md\uff09\n")
print('\nWiki \u6a94\u6848:', sorted(os.listdir('wiki')))

## 步驟二【整理重點 → 檢查內容】先討論、後生成

討論結論（寫入各篇 Wiki 的「loop/graph 角色」段落，作為簡報知識來源）：

| 論文 | 在 Agent loop / graph 中的角色 |
|---|---|
| ReAct | 核心循環：思考 \u2192 行動 \u2192 \u89c0察 \u7684 \u8fed\u4ee3\u751f\u5f80\u8fd4\u5faa\u74b0 |
| Reflexion | \u53cd\u601d\u7bc0\u9ede\uff1a\u5931\u6557\u5f8c\u751f\u6210\u6587\u5b57\u56de\u994b\u91cd\u8a66\uff08\u5167\u578b\u5167 loop\uff09 |
| Toolformer | \u5de5\u5177\u884c\u52d5\u7bc0\u9ede\uff1a\u6280\u80fd\u64f4\u5145 Agent \u53ef\u505a\u7684\u4e8b |
| Generative Agents | \u8a18\u5fc6\u67b6\u69cb\uff1a\u89c0\u5bdf\u6c89\u6dc0 \u2192 \u53cd\u601d \u2192 \u8907\u74b0\u7d22\u53d6 |  
| CRAG | \u6aa2\u67e5\u9580\u7981\uff1a\u6aa2\u7d22\u54c1\u8cea\u5206\u7d1a\u8207\u91cd\u65b0\u6574\u7406\uff08\u6821\u9a57\u7bc0\u9ede\u524d\u79fb\uff09 |

In [ ]:
# %%cell【檢查內容】Validator：確認 5 篇 Wiki 都有標題+摘要（執行與檢查紀錄）
ok = True
for name in SOURCES:
    t = open(f'wiki/{name}.md', encoding='utf-8').read()
    has_title = t.startswith('# ') and len(t.split('#')[1]) > 5
    has_abs = '## Abstract' in t and len(t.split('## Abstract')[1]) > 100
    print(f'[check] wiki/{name}.md title={has_title} abstract={has_abs}')
    ok = ok and has_title and has_abs
print('\n=== 檢查結果:', 'PASS：5/5 篇 Wiki 完整，可作為簡報知識庫' if ok else 'FAIL：需要重新 ingest')